# Export LaMa inpainting model to ONNX

Author: [Nikita Selin](https://github.com/OPHoperHPO), [Carve.Photos Team](https://carve.photos) \
HuggingFace Repository with ONNX Model: [Link](https://huggingface.co/Carve/LaMa-ONNX) \
Original repository: [Link](https://github.com/advimman/lama)

---

## ⚠️ Important Notes on ONNX Preprocessing/Postprocessing

**Critical differences between PyTorch and ONNX implementations:**

1. **Input preprocessing**: Images and masks must be normalized to [0, 1] range (divide by 255)
2. **Output postprocessing**: **CRITICAL** - The ONNX model outputs values already in [0, 255] range
   - The model internally multiplies by 255 (see `ExportLama.forward()` below)
   - DO NOT multiply output by 255 again - it's already scaled!
   - Simply convert to uint8: `output.astype(np.uint8)`

3. **Common mistakes**:
   - ❌ Multiplying or dividing output by 255
   - ❌ Not normalizing inputs to [0, 1]
   - ❌ Wrong tensor format (must be CHW with batch dimension)

For detailed documentation, see the README's "ONNX Preprocessing & Postprocessing Guide" section.

## install deps

In [1]:
!git clone https://github.com/Carve-Photos/lama --depth 1

Cloning into 'lama'...
remote: Enumerating objects: 248, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 248 (delta 52), reused 131 (delta 48), pack-reused 0
Receiving objects: 100% (248/248), 5.44 MiB | 17.63 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [2]:
%cd lama

/content/lama


In [3]:
!curl -LJO https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.zip
!unzip big-lama.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1195  100  1195    0     0   5834      0 --:--:-- --:--:-- --:--:--  5857
100  363M  100  363M    0     0   143M      0  0:00:02  0:00:02 --:--:--  165M
Archive:  big-lama.zip
  inflating: big-lama/config.yaml    
  inflating: big-lama/models/best.ckpt  


In [ ]:
!pip3 install omegaconf webdataset pytorch_lightning pytorch_lightning kornia==0.5.0 onnx onnxruntime

## Init model

**Note**: The `ExportLama` class below is crucial for understanding ONNX preprocessing/postprocessing:
- It wraps the LaMa model for ONNX export
- **CRITICAL**: The `forward()` method multiplies output by 255 before returning
- This is why ONNX inference outputs are already in [0, 255] range!

In [ ]:
import torch
from omegaconf import OmegaConf
from yaml import safe_load

from saicinpainting.training.trainers.default import (
    DefaultInpaintingTrainingModule,
)


class ExportLama(torch.nn.Module):
    """
    Wrapper class for exporting LaMa model to ONNX format.
    
    CRITICAL: This class defines the preprocessing/postprocessing behavior of the ONNX model!
    """
    def forward(self, image: torch.Tensor, mask: torch.Tensor):
        """
        Forward pass for ONNX export.
        
        Args:
            image: Input image tensor in range [0, 1], shape (batch, 3, H, W)
            mask: Binary mask tensor in range [0, 1], shape (batch, 1, H, W)
        
        Returns:
            Inpainted image tensor in range [0, 255], shape (batch, 3, H, W)
        
        IMPORTANT: 
        - Input image and mask are expected to be normalized to [0, 1]
        - Output is scaled to [0, 255] range (see the final line: * 255)
        - This is why ONNX model outputs don't need additional scaling!
        """
        # Step 1: Apply mask to image (mask out the region to inpaint)
        masked_img = image * (1 - mask)

        # Step 2: Concatenate mask with masked image if model requires it
        if self.model.concat_mask:
            masked_img = torch.cat([masked_img, mask], dim=1)

        # Step 3: Generate inpainted content using the generator network
        predicted_image = self.model.generator(masked_img)
        
        # Step 4: Blend predicted content with original image
        # Only replace masked regions, keep original image elsewhere
        inpainted = mask * predicted_image + (1 - mask) * image
        
        # Step 5: Scale output to [0, 255] range and clamp
        # ⚠️ CRITICAL: This multiplication by 255 is why ONNX outputs are already scaled!
        return torch.clamp(inpainted * 255, min=0, max=255)

## Export ONNX

This cell performs the actual ONNX export with proper configuration.

In [ ]:
# Load the configuration file
config = OmegaConf.create(safe_load(open("/content/lama/big-lama/config.yaml")))

# Extract and modify training model configuration
kwargs = dict(config.training_model)
kwargs.pop("kind")
kwargs["use_ddp"] = True

# Enable JIT version of FourierUnit, required for ONNX export
config.generator.resnet_conv_kwargs.use_jit = True

# Fix the configuration by setting the weight to zero
config.losses.resnet_pl.weight = 0

# Load the model state
state = torch.load("/content/lama/big-lama/models/best.ckpt", map_location="cpu")
lama_dilated_model = DefaultInpaintingTrainingModule(config, **kwargs)
lama_dilated_model.load_state_dict(state["state_dict"], strict=False)
lama_dilated_model.on_load_checkpoint(state)
lama_dilated_model.freeze()
lama_dilated_model.eval()

# Export the model using the ExportLama wrapper
exported_model = ExportLama()
exported_model.register_module("model", lama_dilated_model)
exported_model.eval()
exported_model.to("cpu")

# Export to ONNX format
# IMPORTANT: The example inputs shown here demonstrate the expected format:
# - Image: shape (1, 3, 512, 512), values in [0, 1] range
# - Mask: shape (1, 1, 512, 512), values in [0, 1] range
torch.onnx.export(
    exported_model,
    (
        # Example image input: batch_size=1, channels=3, height=512, width=512
        # Values are in [0, 1] range (not [0, 255]!)
        torch.rand(1, 3, 512, 512).type(torch.float32).to("cpu"),
        
        # Example mask input: batch_size=1, channels=1, height=512, width=512  
        # Values are in [0, 1] range
        torch.rand(1, 1, 512, 512).type(torch.float32).to("cpu")
    ),
    "/content/lama_fp32.onnx",
    input_names=["image", "mask"],
    output_names=["output"],
    dynamic_axes={
        "image": {0: "batch"},
        "mask": {0: "batch"},
        "output": {0: "batch"}
    },  # Batch dimension is dynamic, but H/W are fixed
    export_params=True,
    do_constant_folding=True,
    opset_version=17,
    verbose=False,
)

print("Lama Model exported to /content/lama_fp32.onnx (open file explorer to download)")
print("\n⚠️ IMPORTANT REMINDER:")
print("- Input images/masks must be in [0, 1] range")
print("- Output will be in [0, 255] range (already scaled!)")
print("- See README for complete preprocessing/postprocessing guide")


## Test exported ONNX model

This section demonstrates the correct preprocessing and postprocessing for ONNX inference.

**Key points demonstrated below:**
1. Input preprocessing: normalize to [0, 1], convert to CHW format, add batch dimension
2. Output postprocessing: output is already in [0, 255], just convert to uint8

In [ ]:
import cv2
import numpy as np
import onnxruntime
import torch
import io
import requests
from PIL import Image

def get_image(image):
    """
    Convert PIL Image or numpy array to model input format.
    
    Returns:
        numpy array in CHW format, normalized to [0, 1] range
    """
    if isinstance(image, Image.Image):
        img = np.array(image)
    elif isinstance(image, np.ndarray):
        img = image.copy()
    else:
        raise Exception("Input image should be either PIL Image or numpy array!")

    # Convert to CHW (Channel, Height, Width) format
    if img.ndim == 3:
        img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
    elif img.ndim == 2:
        img = img[np.newaxis, ...]  # Add channel dimension for grayscale

    assert img.ndim == 3

    # CRITICAL: Normalize to [0, 1] range
    # This is required for ONNX model input!
    img = img.astype(np.float32) / 255
    return img


def ceil_modulo(x, mod):
    """Round up to nearest multiple of mod."""
    if x % mod == 0:
        return x
    return (x // mod + 1) * mod


def scale_image(img, factor, interpolation=cv2.INTER_AREA):
    """
    Scale image by a factor while maintaining CHW format.
    
    Args:
        img: Image in CHW format
        factor: Scaling factor
        interpolation: OpenCV interpolation method
    
    Returns:
        Scaled image in CHW format
    """
    # Convert CHW to HWC for OpenCV
    if img.shape[0] == 1:
        img = img[0]
    else:
        img = np.transpose(img, (1, 2, 0))

    # Resize using OpenCV
    img = cv2.resize(img, dsize=None, fx=factor, fy=factor, interpolation=interpolation)

    # Convert back to CHW
    if img.ndim == 2:
        img = img[None, ...]
    else:
        img = np.transpose(img, (2, 0, 1))
    return img


def pad_img_to_modulo(img, mod):
    """
    Pad image to make dimensions divisible by mod.
    
    This is required because the model expects dimensions divisible by 8.
    Uses symmetric padding to avoid edge artifacts.
    
    Args:
        img: Image in CHW format
        mod: Modulo value (typically 8)
    
    Returns:
        Padded image in CHW format
    """
    channels, height, width = img.shape
    out_height = ceil_modulo(height, mod)
    out_width = ceil_modulo(width, mod)
    return np.pad(
        img,
        ((0, 0), (0, out_height - height), (0, out_width - width)),
        mode="symmetric",  # Symmetric padding to avoid edge artifacts
    )


def prepare_img_and_mask(image, mask, device, pad_out_to_modulo=8, scale_factor=None):
    """
    Prepare image and mask for ONNX model inference.
    
    This function demonstrates the complete preprocessing pipeline:
    1. Convert to CHW format and normalize to [0, 1]
    2. Optionally scale
    3. Pad to required dimensions
    4. Add batch dimension
    5. Binarize mask
    
    Args:
        image: Input image (PIL Image or numpy array)
        mask: Input mask (PIL Image or numpy array)
        device: 'cpu' or 'cuda'
        pad_out_to_modulo: Pad to make dimensions divisible by this value
        scale_factor: Optional scaling factor
    
    Returns:
        Tuple of (image_tensor, mask_tensor) ready for ONNX inference
    """
    # Step 1: Convert to CHW format and normalize to [0, 1]
    out_image = get_image(image)
    out_mask = get_image(mask)

    # Step 2: Scale if requested
    if scale_factor is not None:
        out_image = scale_image(out_image, scale_factor)
        out_mask = scale_image(out_mask, scale_factor, interpolation=cv2.INTER_NEAREST)

    # Step 3: Pad to required dimensions
    if pad_out_to_modulo is not None and pad_out_to_modulo > 1:
        out_image = pad_img_to_modulo(out_image, pad_out_to_modulo)
        out_mask = pad_img_to_modulo(out_mask, pad_out_to_modulo)

    # Step 4: Convert to PyTorch tensor and add batch dimension
    out_image = torch.from_numpy(out_image).unsqueeze(0).to(device)
    out_mask = torch.from_numpy(out_mask).unsqueeze(0).to(device)

    # Step 5: Binarize mask (values > 0 become 1, else 0)
    out_mask = (out_mask > 0) * 1

    return out_image, out_mask

def open_image(image):
    """Load image from URL or file path."""
    if isinstance(image, str):
      if image.startswith("http://") or image.startswith("https://"):
        image = Image.open(io.BytesIO(requests.get(image).content))
      else:
        image = Image.open(image)
    return image

In [ ]:
# Initialize ONNX Runtime session
sess_options = onnxruntime.SessionOptions()
model = onnxruntime.InferenceSession('/content/lama_fp32.onnx', sess_options=sess_options)

In [ ]:
#@title Predict with ONNX model
image_url = "https://huggingface.co/Carve/LaMa-ONNX/resolve/main/image.jpg" # @param {type:"string"}
mask_url = "https://huggingface.co/Carve/LaMa-ONNX/resolve/main/mask.png" # @param {type:"string"}

# Load test images
image = open_image(image_url).resize((512, 512))
mask = open_image(mask_url).convert("L").resize((512, 512))

# Preprocess inputs
# This converts to CHW format, normalizes to [0, 1], pads, and adds batch dimension
image, mask = prepare_img_and_mask(image, mask, 'cpu')

# Run ONNX inference
# IMPORTANT: Inputs are numpy arrays with:
# - image: shape (1, 3, H, W), dtype float32, range [0, 1]
# - mask: shape (1, 1, H, W), dtype float32, range [0, 1]
outputs = model.run(None,
                    {'image': image.numpy().astype(np.float32),
                     'mask': mask.numpy().astype(np.float32)})

# Get the output (first element of outputs list, remove batch dimension)
output = outputs[0][0]

# Postprocess the output
# CRITICAL: The output is already in [0, 255] range!
# We only need to:
# 1. Convert from CHW to HWC format
# 2. Convert to uint8
output = output.transpose(1, 2, 0)  # CHW -> HWC
output = output.astype(np.uint8)    # Convert to uint8 (NO scaling needed!)

# Convert to PIL Image for display
output = Image.fromarray(output)
output